# What happens at the `synthesise` step

`synthesise` is the last component of every Evidence search run. Everything before it finds, filters and
reads documents. `synthesise` is the step that **writes the report** and **checks every sentence of it
against the sources**.

This notebook explains the step in plain English and shows each part on a **real production run**, read
from Langfuse (our tracing system, which records every model call with its full input and output).
Nothing here writes to Langfuse or to the database: the notebook only reads one finished trace.

**The whole step in one paragraph.** The writer model is given the user's question and a summary of what
the earlier steps produced. It first proposes the section headings. Then, one section at a time, it goes
to the "library" (three read-only tools over the screened-in documents and the extracted findings), reads
what it needs, and hands in the section as prose plus a list of *claims*: every factual sentence, typed
and tied to the exact source it rests on. Code then checks each claim mechanically (is the quote really in
the source? is the number right?), and a second, cheaper model (the *grounding judge*) grades each cited
claim for how well the source supports it. Claims that fail get one chance to be rewritten. After the body
sections, the same machinery writes the Conclusions, the Key findings, Case studies, a few source notes
and an introduction. Finally the report and every claim, citation and verdict are saved, and one-line
navigation summaries are written and judged.

**Roadmap**

| § | Stage | Who does the work |
|---|-------|-------------------|
| 1 | The big picture: what goes in, what comes out | — |
| 2 | Propose the section headings | 1 LLM call (writer model) |
| 3 | Write each section with the tool loop | up to 6 LLM turns per section + tool reads |
| 4 | The six claim types | — |
| 5 | Deterministic checks on every claim | code, no LLM |
| 6 | The grounding judge | 1 LLM call per section (judge model) |
| 7 | One repair pass | 1 LLM call per section that had failures, then a re-judge |
| 8 | After the body: Conclusions, Key findings, Case studies, notes, intro | LLM calls |
| 9 | What is saved, and the roll-up | code |
| 10 | Navigation summaries, after the commit | LLM writer + LLM judge |
| 11 | Cost and time of the run | — |
| 12 | Glossary | — |

**Before you start.** Run this notebook on the `Policy Atlas (backend)` kernel. `backend/.env` must
contain `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY` and `LANGFUSE_HOST`. Traces contain the full
prompts, including text from third-party documents, so keep the outputs inside Nesta.

**Where the code lives** (for reference only): `backend/src/policy_atlas/evidence_search/synthesis/`.
`synthesise.py` is the orchestrator (`synthesise_scope`), `synthesis_backend.py` holds the writer prompts
and the model calls, `synthesis_tools.py` holds the read tools and the loop, `grounding_judge.py` the judge,
`summary_prompts.py` the summaries.

## 0. Setup: connect to Langfuse and load one real run

A Langfuse *trace* is one pipeline run. Inside it, every model call and every tool call is an
*observation*. The names are stable, so we can find each stage by name:

| Observation name | What it is |
|---|---|
| `synthesise:propose_sections` | the section-heading proposal (§2) |
| `synthesise:section_turn` | one turn of the writer inside the section loop (§3) |
| `tool:search_chunks`, `tool:query_findings`, `tool:lookup` | the reads the writer asked for (§3) |
| `embed:batch` | embedding the search queries so `search_chunks` can rank by meaning |
| `judge:grounding` | the grounding judge grading one block's claims (§6) |
| `synthesise:repair_claims` | the one repair call (§7) |
| `synthesise:write_key_findings`, `synthesise:write_case_studies` | the two front-matter blocks (§8) |
| `synthesise:write_source_note`, `synthesise:write_intro` | source notes and the report intro (§8) |
| `synthesise:write_block_summary`, `synthesise:write_artefact_summary`, `synthesise:judge_summary` | summaries and their judge (§10) |

In [ ]:
import json
import os
import textwrap
from collections import Counter
from datetime import datetime, timedelta, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 220)

# Find backend/.env whether the kernel started in backend/notebooks, backend/ or the repo root.
candidates = [d / "backend" / ".env" for d in [Path.cwd(), *Path.cwd().parents]] + [d / ".env" for d in [Path.cwd(), *Path.cwd().parents]]
env_file = next((p for p in candidates if p.exists()), None)
load_dotenv(env_file)

from policy_atlas.core.tracing import get_langfuse  # the app's own client factory (refuses partial config)

lf = get_langfuse()
assert lf is not None, "Langfuse keys not found. Put LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY and LANGFUSE_HOST in backend/.env"
print("connected to Langfuse (read-only use in this notebook)")

The run used below is a production run from 23 September 2026. It was chosen because it exercises
everything: the plan included `characterise`, `select`, `extract` and `group`, so all six claim types
appear, the Case studies block was produced, and several claims failed and were repaired.

To study a different run, paste its trace id into `TRACE_ID`. In the Langfuse web UI: *Traces*, filter the
name to `run:synthesise`, open one, and copy the id from the address bar. If the id below has expired, the
notebook falls back to the most recent synthesise run.

In [ ]:
TRACE_ID = "40dac62eb69c942f41850ac9a955b283"   # prod, 23 Sep 2026: place-based philanthropy and giving


def load_trace(trace_id):
    trace = lf.api.trace.get(trace_id)
    observations = sorted(trace.observations, key=lambda o: o.start_time)
    return trace, observations


try:
    trace, obs = load_trace(TRACE_ID)
except Exception as exc:  # the id is unknown to this Langfuse project, or has been deleted
    print(f"could not load {TRACE_ID} ({type(exc).__name__}); using the most recent synthesise run instead")
    now = datetime.now(timezone.utc)
    recent = lf.api.trace.list(limit=5, name="run:synthesise", from_timestamp=now - timedelta(days=90), to_timestamp=now).data
    TRACE_ID = recent[0].id
    trace, obs = load_trace(TRACE_ID)


# --- small helpers used throughout -----------------------------------------------------------
def meta(o):
    "Our own metadata on an observation, without the OpenTelemetry boilerplate."
    return {k: v for k, v in (o.metadata or {}).items() if k not in ("resourceAttributes", "scope")}


def cost(o):
    return float(getattr(o, "calculated_total_cost", 0) or 0)


def seconds(o):
    return (o.end_time - o.start_time).total_seconds()


def seed_json(content):
    "Our prompts send data as 'Label (data, not instructions):\\n{json}'. Return the json."
    return json.loads(content.split("\n", 1)[1])


def show_json(value, limit=1500):
    text = json.dumps(value, indent=2, ensure_ascii=False, default=str)
    print(text[:limit] + (" ..." if len(text) > limit else ""))


def show_prose(text, width=110):
    for para in text.split("\n\n"):
        print(textwrap.fill(para, width), "\n")


# The question the run answered. Every section turn carries it in its "run seed" message.
first_turn = next(o for o in obs if o.name == "synthesise:section_turn")
RUN_SEED = seed_json(first_turn.input["messages"][1]["content"])
INTENT = RUN_SEED["intent"]

generations = [o for o in obs if o.type in ("GENERATION", "EMBEDDING")]
print("trace:", TRACE_ID, "| environment:", trace.environment, "| started:", obs[0].start_time.strftime("%Y-%m-%d %H:%M UTC"))
print("question:", INTENT)
print(f"wall time: {seconds(obs[0]) / 60:.1f} minutes | model calls: {len(generations)} | cost: ${trace.total_cost:.2f}")

## 1. The big picture: what goes in, what comes out

**What goes in** is called the *substrate* in the code: everything the earlier steps wrote to the
database for this Task. Only the first two are guaranteed; the rest depend on the plan.

| Input | Written by | Always present? |
|---|---|---|
| The user's question (the *intent*) and any steering (a section list, a section budget, retrieval boosts) | planner | yes |
| The screened-in documents, each as *frozen text chunks*: the full text where it was fetched, otherwise the abstract. Each chunk carries whether its document was appraised and whether the text is `full_text` or `abstract_only`. | acquire, screen, ingest | yes |
| Classification (document type) and appraisal (quality score 1 to 5) per document | classify, appraise | yes |
| The characterisation: coverage counts and themes (the landscape) | characterise | only at standard and detailed depth |
| The selection: which documents were chosen for the expensive deep read | select | only at standard and detailed depth |
| Extracted findings: IOF (intervention–outcome findings: "X changed Y in direction Z") and ICF (implementation-context findings: barriers, enablers, mechanisms), each with verified anchor quotes | extract | only at detailed depth |
| Facet groups: the findings clustered by intervention, outcome, barrier theme, and so on | group | only at detailed depth |

If there is nothing groundable at all (no references and zero screened-in documents) the step refuses
to run with the error `no_groundable_substrate`. A run with only the mandatory inputs still produces a
report: a "grounded answer" with chunk claims, but no themes or findings.

**What comes out** is one *artefact* (the report) made of *blocks* (one per section, plus Key findings
and Case studies). Each block is prose plus:

- **claims**: the evidential sentences, each typed (§4) and located as an exact character span of the prose;
- **citations**: for cited claims, the quote and the chunk it was found in, with the deterministic check result;
- **verdicts**: for judged claims, the grounding tier, the "weakly grounded" flag and a one-sentence rationale;
- **flags**: anything that went wrong is recorded, never hidden ("flag, don't drop").

A roll-up row (`synthesis_result`) records counts, flags and full provenance (prompt versions, model names,
caps, which upstream runs were used). The trace's output is that roll-up, so let us read it.

In [ ]:
out = trace.output
print("which upstream components this run could draw on:")
display(pd.Series(out["substrate_profile"]).to_frame("value"))
print("claims that survived, by type, and by judge verdict:")
display(pd.DataFrame({"by type": out["counts"]["claims_total"]}).T)
display(pd.DataFrame({"by verdict": out["counts"]["claims_by_verdict_lane"]}).T)
print("flags raised on this run:", out["flags"])

Read the two tables with §4 and §6 in mind: *finding* and *chunk* claims are the cited ones; *pattern*,
*theme* and *gap* are checked by code; *reasoning* is labelled background. The verdict lanes are the
judge's grades: tier 1 is a direct quote, tier 4 is labelled reasoning, and `unsupported_mis_cited` means
the source did not support the sentence as written. Those unsupported claims are still in the report,
visibly flagged. Every flag in the last line is explained in §9.

Now the shape of the run as a sequence of model calls. The counts tell the story on their own: one
proposal, two writer turns per section, one judge per section, a repair where needed.

In [ ]:
rows = []
for name, group in pd.DataFrame({"o": generations}).assign(name=lambda f: f["o"].map(lambda o: o.name)).groupby("name"):
    os_ = list(group["o"])
    rows.append({
        "observation": name,
        "calls": len(os_),
        "model": Counter(o.model for o in os_).most_common(1)[0][0],
        "prompt tokens": sum(int(meta(o).get("prompt_tokens") or 0) for o in os_),
        "cost $": round(sum(cost(o) for o in os_), 3),
        "first seen": min(o.start_time for o in os_).strftime("%H:%M:%S"),
    })
pd.DataFrame(rows).sort_values("first seen").reset_index(drop=True)

Two models do the work. The **writer** (`gpt-5.6-terra` at the time of writing) proposes sections,
writes them, writes the Key findings and Case studies, and writes and judges summaries. The **judge**
(`gpt-5.4-mini`) grades claims and never writes anything. The two small "mini" calls (source notes and the
report intro) also use the cheaper model. The embedding model only turns search queries into vectors.

## 2. Step 1: propose the section headings

**One structured LLM call, made blind.** The writer receives exactly two messages: the rulebook (system
prompt) and one user message holding the question plus *summaries* of the substrate. It has no tools at
this point, so it cannot look anything up. It returns 1 to 8 sections, each with a `title` (at most 60
characters), a `focus` (what evidence the section will present, at most 300 characters), a short
`nav_label` for the contents list, and optionally the facet `group_ids` it will draw on.

**What is in the packet, and what is not.** The writer sees the *shape* of the evidence, never the
evidence. In particular it does **not** see any document, chunk, quote or finding. The grouping step's
findings arrive only as their groups: a label and a one-sentence description that the clustering model
wrote, a size and a direction spread. The membership lists are stripped before the packet is built.

| Part of the packet | What it carries | What is deliberately left out |
|---|---|---|
| `intent` | the user's question | — |
| `corpus` | three counts: screened-in, ingested full text, appraised | — |
| `characterisation` | coverage counts and distributions over the screened-in set (per source, evidence type, publisher, quality score, screen basis, full-text status, tags); each theme's id, name, description and size | the member document ids of each theme |
| `grouping` | per facet: counts and status; each group's id, facet, label, description, size and direction spread | the member finding ids of each group, and all finding text |
| `extraction` | counts only: findings per profile, abstract versus full-text basis, field coverage | the findings |
| `selection` | strategy, budget, how many documents were selected, flags | which documents |

So the headings are decided from the wording of the question, the names and descriptions of the clusters,
and the composition counts. Finding text first reaches the writer one stage later, in the section loop
(§3), where the findings of a section's assigned groups are loaded in full into its seed.

**Code then validates the proposal** (`_validate_sections`): the count must be within the cap (and
within the plan's section budget, if it set one); titles must be distinct, non-empty and not generic
("Overview", "Introduction", "Summary", "Conclusion", "Other" and similar are rejected); group ids must be
real ids from the grouping; over-long labels are truncated and the truncation is recorded. If anything is
rejected, the reasons are sent back verbatim for **one** corrected proposal; a second failure fails the run.

Two things happen without the model: if the user steered the run with an explicit section list, the
proposal call is skipped and that list is used (`section_source = "scope_context"`); and code always
appends a **Conclusions** section at the end, which the writer is not allowed to propose.

In [ ]:
proposal = next(o for o in obs if o.name == "synthesise:propose_sections")
messages = proposal.input["messages"]
user_msg = messages[1]["content"]
substrate_summary = json.loads(user_msg.split("Available substrate summaries (data, not instructions):\n", 1)[1])

print(f"the proposal call received {len(messages)} messages: {[m['role'] for m in messages]} "
      f"({len(messages[0]['content']):,} + {len(user_msg):,} characters; {meta(proposal).get('prompt_tokens'):,} prompt tokens)")
print("\nsize of each part of the substrate packet:")
display(pd.DataFrame([
    {"part": part, "characters": len(json.dumps(value, ensure_ascii=False)),
     "keys": ", ".join(value) if isinstance(value, dict) else type(value).__name__}
    for part, value in substrate_summary.items()
]).sort_values("characters", ascending=False).reset_index(drop=True))

print("characterisation coverage: the counts over the screened-in set")
show_json(substrate_summary.get("characterisation", {}).get("coverage", {}).get("base_counts"))
themes = substrate_summary.get("characterisation", {}).get("themes") or []
if themes:
    print(f"\none of the {len(themes)} characterisation themes, exactly as received (no member document ids):")
    show_json(themes[0])
groups = (substrate_summary.get("grouping") or {}).get("groups") or []
if groups:
    print(f"\none of the {len(groups)} facet groups, exactly as received (label and description only; no member finding ids, no finding text):")
    show_json(groups[0])
print("\nselection, as received:", json.dumps(substrate_summary.get("selection"), ensure_ascii=False))
print("extraction, as received (counts only):", json.dumps(substrate_summary.get("extraction"), ensure_ascii=False)[:200], "...")

A direct check that no finding text reached this call. Section 0's seed (§3) carries its member findings in
full; the first of them is a sentence the model could only have seen if findings were in the packet.

In [ ]:
section0_seed = seed_json(first_turn.input["messages"][2]["content"])
member = section0_seed["member_findings"][0] if section0_seed["member_findings"] else None
finding_text = (member or {}).get("claim") or (member or {}).get("intervention") or ""

print("membership ids anywhere in the proposal packet? ", any(key in user_msg for key in ("member_finding_ids", "member_ids")))
print("that finding's text in the proposal packet?      ", bool(finding_text) and finding_text in user_msg)
print("that finding's text in section 0's seed?         ", bool(finding_text) and finding_text in first_turn.input["messages"][2]["content"])
print("\nthe finding in question:", textwrap.shorten(finding_text, 160))

The sections it proposed from that packet:

In [ ]:
print(f"proposal call: {seconds(proposal):.1f} s, ${cost(proposal):.3f}")
pd.DataFrame(proposal.output["sections"]).assign(group_ids=lambda f: f["group_ids"].map(lambda g: ", ".join(g)))

Notice that the titles name an aspect ("Resident power in local grantmaking") rather than restating the
question, and that they read like the group labels. That is no accident: labels and descriptions are all
the writer has at this point. Several sections are assigned facet groups, and those group ids matter in
§3: the findings in an assigned group are pre-loaded into the section's seed, so the writer can cite them
without a tool call. The structure chosen here therefore decides what each section reads first.

**Who sees what, stage by stage.** Each model call in this step gets a different, deliberately limited
packet. Keeping this table in mind resolves most "how did it know that?" questions.

| Call | Sees | Does not see |
|---|---|---|
| propose sections (§2) | question; summaries: counts, theme and group labels and descriptions, sizes, direction spreads | documents, chunks, findings, quotes |
| section loop (§3) | the above, plus this section's title and focus, the **full findings** of its assigned groups, the ledger of earlier sections' claim texts, and whatever its tool reads return (chunk text, finding records, lookups) | other sections' tool reads; the documents beyond what it read |
| grounding judge (§6) | the claims, the full frozen text of every cited chunk (and of the chunks the findings' anchors live in), the section prose, the span map, the question, the section focus | the tool transcript; anything the writer read but did not cite |
| repair (§7) | the run seed, the failing claims with their failure reasons, the prose segment and its neighbours, and the dependency records (the cited chunks' and findings' full content) | the tool transcript; new reads |
| key findings and case studies (§8) | the ledger of surviving claims of every section with verdicts, finding ids and quotes, plus the cited chunks' text (case studies also get each cited document's appraisal, type and year) | anything that did not survive verification |
| summaries (§10) | the block's prose with its claims' epistemic status marked | sources and citations |

The full rulebook the model follows for the proposal call:

In [ ]:
from policy_atlas.evidence_search.synthesis.synthesis_backend import SECTIONS_PROMPT_VERSION, SECTIONS_SYSTEM_PROMPT

print("prompt version:", SECTIONS_PROMPT_VERSION, "\n")
print(SECTIONS_SYSTEM_PROMPT)

A small demonstration of the validator, with a deliberately bad proposal. The reasons it returns are
written as instructions because they are fed straight back to the model for the one repair attempt.

In [ ]:
from policy_atlas.evidence_search.synthesis.synthesis_backend import SectionProposalWire
from policy_atlas.evidence_search.synthesis.synthesise import CONCLUSIONS_TITLE, _conclusions_focus, _validate_sections

bad = SectionProposalWire.model_validate({"sections": [
    {"title": "Overview", "focus": "Everything.", "nav_label": "Overview"},
    {"title": "What the evidence shows about place-based philanthropy and giving in England", "focus": "A title that restates the question and is too long.", "nav_label": "Long title"},
    {"title": "Resident power in local grantmaking", "focus": "Fine.", "nav_label": "Resident power", "group_ids": ["intervention:g99"]},
]})
sections, reasons, normalisations = _validate_sections(bad, grouping_group_ids={"intervention:g04", "outcome:g01"})
print("rejection reasons sent back to the model:")
for r in reasons:
    print(" -", r)

print("\nThe section code appends last, with a focus derived from the question:")
print(" ", CONCLUSIONS_TITLE, "->", _conclusions_focus(INTENT))

## 3. Step 2: write each section with the tool loop

This is the heart of the step, and the only place where the model behaves like an *agent*: it decides
what to read before it writes. Sections are written **one after another**, so each section sees a
*ledger* of the claims the earlier sections already made (to avoid repetition, not as evidence).

**How one section goes.**

1. Code builds the *seed*: the question, this section's title and focus, the substrate summaries, the tools
   and claim types available on this run, the member findings of any assigned groups (with their computed
   direction spread), and the ledger.
2. **Turn 1.** The writer either asks for reads (up to 6 tool calls in one turn) or hands in the section.
   The reads run; their results are appended to the conversation as data.
3. **Turns 2 to 6.** Same choice each turn. On turn 6 the writer is told it is the final turn and must hand
   in. Hand-in is itself a tool call, `emit_section`, which carries the prose and the claims.
4. The conversation is rebuilt from scratch every turn (system prompt, run seed, section seed, then every
   tool exchange), so the model is stateless and most of the prompt is served from the provider's cache.

**The three read tools.** All are read-only, scoped to this Task, and validate every argument (an unknown
id or vocabulary value is an error the model sees and can correct).

| Tool | What it reads | Notes |
|---|---|---|
| `search_chunks` | the frozen text chunks of every **screened-in** document, ranked by meaning (embedding) and by keywords, fused | 8 chunks per call and a 24,000-character read budget per section. Selected documents get a ranking boost but unselected screened-in documents are still readable; every chunk says which (`origin`). Only chunks of **appraised** documents may be cited. |
| `query_findings` | the extracted findings: `iof_findings` and `icf_findings` in separate lists | only exists when the run included `extract`. Filters: group id, finding ids, effect direction (IOF only), context type (ICF only). |
| `lookup` | canonical facts: appraisals, classifications, selection rationale, search coverage records, the characterisation summary, the grouping groups, tags | deterministic reads of database rows |

Two markers keep the conversation small: a record already returned earlier in the section comes back as
`{"id": ..., "already_returned": true}` (still citable: the model has its content above), and a chunk that
would overflow the section's read budget comes back as `{"id": ..., "skipped_over_budget": true}` (not
citable: its content was never shown).

In [ ]:
from policy_atlas.evidence_search.synthesis.synthesis_tools import (
    READ_CALLS_PER_TURN_CAP, REASONING_CLAIMS_MAX, SECTION_CAP, SECTION_TURN_CAP, SYNTH_CHUNK_CHAR_BUDGET, SYNTH_CHUNK_TOP_K,
)

print(f"caps: {SECTION_CAP} proposed sections (+ Conclusions) | {SECTION_TURN_CAP} turns per section | {READ_CALLS_PER_TURN_CAP} reads per turn | "
      f"{SYNTH_CHUNK_TOP_K} chunks per search | {SYNTH_CHUNK_CHAR_BUDGET:,} characters of chunk text per section | {REASONING_CLAIMS_MAX} reasoning claims per section")
print("tools and claim types available on this run:", RUN_SEED["available_tools"], RUN_SEED["available_claim_types"])

To read the trace section by section we split the observations into phases. A `synthesise:section_turn`
whose `transcript_length` is 0 is the first turn of a new section; the later passes start at their own
named observation.

In [ ]:
def split_phases(observations):
    phases, current, n = [], {"name": "proposal", "obs": []}, 0
    phases.append(current)
    for o in observations:
        start = None
        if o.name == "synthesise:section_turn" and meta(o).get("transcript_length") == 0:
            start, n = f"section {n}", n + 1
        elif o.name == "synthesise:write_key_findings":
            start = "key findings"
        elif o.name == "synthesise:write_case_studies":
            start = "case studies"
        elif o.name == "synthesise:write_source_note" and current["name"] != "source notes":
            start = "source notes"
        elif o.name == "synthesise:write_intro":
            start = "report intro"
        elif o.name.endswith(("_summary",)) and current["name"] != "summaries":
            start = "summaries"
        if start is not None:
            current = {"name": start, "obs": []}
            phases.append(current)
        current["obs"].append(o)
    return {p["name"]: p["obs"] for p in phases}


def section_seed(turn):
    "The section seed is the third message of every section turn."
    return seed_json(turn.input["messages"][2]["content"])


def emission_of(turn):
    "The hand-in, if this turn was the emit_section turn."
    return (turn.output or {}).get("claims")


phases = split_phases(obs)
rows = []
for name, os_ in phases.items():
    if not name.startswith("section"):
        continue
    turns = [o for o in os_ if o.name == "synthesise:section_turn"]
    seed = section_seed(turns[0])
    emitted = next((emission_of(t) for t in turns if emission_of(t)), None)
    rows.append({
        "phase": name,
        "title": seed["section"]["title"],
        "groups": len(seed["section"]["group_ids"]),
        "seeded findings": len(seed["member_findings"]),
        "turns": len(turns),
        "reads": dict(Counter(o.name.split(":")[1] for o in os_ if o.name.startswith("tool:"))),
        "claims emitted": dict(Counter(c["claim_type"] for c in emitted["claims"])) if emitted else None,
        "judge calls": sum(o.name == "judge:grounding" for o in os_),
        "repairs": sum(o.name == "synthesise:repair_claims" for o in os_),
        "cost $": round(sum(cost(o) for o in os_), 2),
        "seconds": round((os_[-1].end_time - os_[0].start_time).total_seconds()),
    })
pd.DataFrame(rows)

On this run every section used two turns: one turn of batched reads, one hand-in. The last row is the
code-injected Conclusions section; it has no groups and runs through exactly the same loop. "Judge calls"
is 2 where a repair happened and at least one repaired claim needed re-judging (initial judge, then a
re-judge); it stays 1 when the repair only deleted or de-claimed segments, which leaves nothing to judge.

### One section, turn by turn

Change `SECTION` to walk through another one.

In [ ]:
SECTION = "section 0"
sec_obs = phases[SECTION]
turns = [o for o in sec_obs if o.name == "synthesise:section_turn"]
seed = section_seed(turns[0])

print("title :", seed["section"]["title"])
print("focus :", seed["section"]["focus"])
print("groups:", seed["section"]["group_ids"])
print("member findings pre-loaded from those groups:", len(seed["member_findings"]), "| computed direction spread:", seed["computed_spread"])
print("ledger entries from earlier sections:", len(seed["ledger"]))
if seed["member_findings"]:
    print("\none pre-loaded member finding (the writer can cite its finding_id without a tool call):")
    show_json(seed["member_findings"][0], 900)

In [ ]:
# Reads in one turn run in a different order from the model's list, so match each call to its result by its arguments.
results_by_call = {}
for o in (o for o in sec_obs if o.name.startswith("tool:")):
    results_by_call.setdefault((o.name.split(":")[1], json.dumps(o.input, sort_keys=True)), []).append(o)

for i, turn in enumerate(turns, 1):
    m, result = meta(turn), turn.output or {}
    print(f"=== turn {i}: {m.get('prompt_tokens'):,} prompt tokens (cached: {m.get('cached_tokens'):,}), {seconds(turn):.0f} s"
          + ("  [final turn: hand-in forced]" if m.get("force_emit") else ""))
    if result.get("claims"):
        print(f"    -> emit_section: {len(result['claims']['prose']):,} characters of prose, {len(result['claims']['claims'])} claims")
        continue
    for call in result.get("tool_calls", []):
        print(f"    -> {call['tool']}({json.dumps(call['arguments'], ensure_ascii=False)[:150]})")
        matches = results_by_call.get((call["tool"], json.dumps(call["arguments"], sort_keys=True)), [])
        if not matches:
            print("       (rejected before execution: unknown tool or invalid arguments; the error went back to the model as data)")
            continue
        tool = matches.pop(0)
        res = tool.output or {}
        if tool.name == "tool:search_chunks":
            chunks = res.get("chunks", [])
            full = [c for c in chunks if "content" in c]
            print(f"       returned {len(chunks)} records: {len(full)} with text, "
                  f"{sum(1 for c in chunks if c.get('already_returned'))} already returned, {sum(1 for c in chunks if c.get('skipped_over_budget'))} skipped over budget")
            if full:
                c = full[0]
                print(f"       first: '{c['document_title'][:70]}' | origin={c['origin']} appraised={c['appraised']} text_basis={c['text_basis']} "
                      f"type={c.get('evidence_type')} quality={c.get('appraisal_label')}")
                print("       content starts:", repr(c["content"][:160]))
        elif tool.name == "tool:query_findings":
            iof, icf = res.get("iof_findings") or [], res.get("icf_findings") or []
            print(f"       returned {len(iof)} IOF and {len(icf)} ICF findings")
            f = (icf or iof)[0] if (icf or iof) else None
            if f:
                print(f"       first ({f['kind']}): {f.get('claim') or (f.get('intervention') + ' -> ' + str(f.get('outcome')) + ' (' + str(f.get('effect_direction')) + ')')!s:.150}")
                print(f"       from '{f['document_title'][:70]}' | context_type={f.get('context_type')} | quality={f.get('appraisal_label')}")
        else:
            print("       returned keys:", list(res.get("result", res).keys())[:8] if isinstance(res.get("result", res), dict) else type(res.get("result")).__name__)

The hand-in. The prose is what a reader sees. Every claim's `text` is copied character for character from
that prose: the model is never asked for character offsets, code locates the span (§5).

In [ ]:
emitted = next(emission_of(t) for t in turns if emission_of(t))
show_prose(emitted["prose"])

claims_table = pd.DataFrame([
    {
        "#": i,
        "type": c["claim_type"],
        "text": c["text"][:95] + ("…" if len(c["text"]) > 95 else ""),
        "cites": (f"{len(c['cited_finding_ids'])} finding(s)" if c["cited_finding_ids"] else "")
                 + (f"{len(c['citations'])} quote(s)" if c["citations"] else "")
                 + (f"pattern over {c['pattern']['computed_from']}" if c.get("pattern") else "")
                 + (f"theme {c['theme']['referenced_ids']}" if c.get("theme") else "")
                 + (f"gap: {c['gap']['grade']}" if c.get("gap") else "")
                 or "(labelled reasoning)",
        "first quote": (c["citations"][0]["quote"][:70] + "…") if c["citations"] else "",
    }
    for i, c in enumerate(emitted["claims"])
])
claims_table

### What the writer actually receives

The conversation for the hand-in turn, message by message. The first three messages never change within a
run (system prompt) or within a section (run seed, section seed), which is what makes the provider cache
effective; the tool exchanges accumulate after them.

In [ ]:
final_turn = next(t for t in turns if emission_of(t))
pd.DataFrame([
    {"role": m["role"],
     "starts with": (m.get("content") or f"[tool call: {m['tool_calls'][0]['function']['name']}]")[:100].replace("\n", " "),
     "characters": len(m.get("content") or json.dumps(m.get("tool_calls")))}
    for m in final_turn.input["messages"]
])

And the rulebook itself: the system prompt of the section loop. It is long because it carries the house
style (the voice principles), the rules of the tool loop, and the rules for each claim type. It is pinned
by a hash so that any edit is a deliberate, recorded change.

In [ ]:
from policy_atlas.evidence_search.synthesis.synthesis_backend import SECTION_PROMPT_VERSION, SECTION_SYSTEM_PROMPT, SECTION_TOOL_SCHEMAS

print("prompt version:", SECTION_PROMPT_VERSION)
print("tools offered to the model:", [t["function"]["name"] for t in SECTION_TOOL_SCHEMAS], "\n")
print(SECTION_SYSTEM_PROMPT)

## 4. The six claim types

A *claim* is one evidential sentence (or clause) of the prose, with a type that says what kind of support
it has. The type decides what the claim must carry, how code checks it, and whether the judge sees it.
The sentences *between* claims are "connective tissue" and must not assert anything about the evidence;
the judge flags it when they do (§6).

| Type | In plain English | Must carry | Code check (§5) | Judged by the LLM (§6)? |
|---|---|---|---|---|
| `finding` | "an extracted finding says X" | ids of findings that were in the seed or returned by `query_findings` **in this section** | ids are real and were readable; the finding's own *verified anchor quotes* are attached by the system (the writer never writes quotes for findings) | yes |
| `chunk` | "a source document says X", with a verbatim quote | chunk id + exact quote | chunk returned in this section; its document appraised; quote found in the frozen text | yes |
| `pattern` | a count or direction spread ("eleven of fifteen…") | where it was computed from, and the numbers stated | numbers recomputed from the substrate; any mismatch rejects | no |
| `theme` | "the clustering groups these together" | theme ids or facet group ids | ids exist in the referenced characterisation or grouping | no |
| `gap` | "nothing was found about X" | a grade (`corpus_absence`, `acknowledged_sparsity`, `inferred`) and a coverage base | corpus absence needs a non-inadequate search coverage record, otherwise it is downgraded to inferred and flagged; a sparsity count must match the characterisation | no |
| `reasoning` | visibly labelled background reasoning | nothing | at most 3 per section; no citations allowed | yes, strictly: it must not smuggle in empirical content |

Which types are *available* depends on the substrate: `finding` needs an extraction, `theme` needs a
characterisation or a grouping, `pattern` needs one of those too. The seed tells the model which types it
may use; any other type is rejected. One real example of each type from this run:

In [ ]:
examples = {}
for name, os_ in phases.items():
    if not name.startswith("section"):
        continue
    for t in (o for o in os_ if o.name == "synthesise:section_turn"):
        for c in (emission_of(t) or {"claims": []})["claims"]:
            examples.setdefault(c["claim_type"], (name, c))

for claim_type in ("finding", "chunk", "pattern", "theme", "gap", "reasoning"):
    if claim_type not in examples:
        print(f"### {claim_type}: not used on this run\n")
        continue
    name, c = examples[claim_type]
    payload = {k: v for k, v in c.items() if v not in ([], None)}
    print(f"### {claim_type}  (from {name})")
    show_json(payload, 1100)
    print()

Read the `pattern` example closely: the text says "eleven extracted outcome findings record increases" but
the typed payload states `increase: 5` for group `outcome:g01`. Code recomputes the direction spread of
that group and requires equality with the *stated* numbers, so the typed numbers are what gets verified.
The `gap` example is graded `inferred` and names its coverage base; an `inferred` gap is always legal but
is visibly labelled as the writer's inference, never as proven absence.

## 5. Step 3: deterministic checks on every claim (code, no LLM)

Before any judge sees a claim, code runs the checks that need no judgement. They run in this order.

**1. Span binding.** Each claim's `text` must occur in the prose, and claims may not overlap. Binding
walks the prose with a cursor so that a repeated sentence binds to its next occurrence. A claim whose text
is not found fails with `span_not_found` and goes to the repair lane. Because the published report locates
every claim by its span, a claim whose text differs from the prose by one character cannot be shown.

In [ ]:
from policy_atlas.evidence_search.synthesis.synthesise import bind_spans

prose = "Payroll giving has declined to £131 million. Only around 4,000 employers offer a scheme. Only around 4,000 employers offer a scheme."
texts = [
    "Payroll giving has declined to £131 million.",      # binds
    "Only around 4,000 employers offer a scheme.",       # binds to the first occurrence
    "Only around 4,000 employers offer a scheme.",       # binds to the second occurrence
    "Payroll giving has declined to £131m.",             # not an exact substring -> None
    "giving has declined to £131 million.",              # overlaps the first bound span -> None
]
for text, span in zip(texts, bind_spans(prose, texts)):
    print(f"{str(span):12} {text!r}" + (f"  -> prose[{span[0]}:{span[1]}] == text: {prose[span[0]:span[1]] == text}" if span else ""))

**2. Per-type rules.** Then each claim is checked by the rules in the §4 table. For a `chunk` claim the
important check is the **quote-presence check**: the quote must occur in the document's frozen text.
The match is tolerant of things a copy can change without changing meaning (letter case, runs of
whitespace, curly versus straight quotes, en and em dashes, soft hyphens), and nothing else. The result is
graded `exact`, `normalised` or `failed`, and the check records *where* the quote was found. That recorded
location, not the model's citation, is what the report links to. A quote that spans two chunks yields two
spans.

In [ ]:
from policy_atlas.evidence_search.extract.quote_verify import QuoteMatcher, build_basis

# A document's frozen text as two chunks (the real basis is every chunk of the document, in order).
basis = build_basis([
    ("chunk-A", "Payroll giving has declined to £131 million despite 30 million PAYE workers being eligible.\nThe scheme is “use it or lose it”."),
    ("chunk-B", "Only around 4,000 employers offer schemes although more than 45,000 could do so."),
])
matcher = QuoteMatcher(basis)
for quote in [
    "declined to £131 million despite 30 million PAYE workers",          # verbatim
    "declined to £131 million   despite 30 MILLION paye workers",        # case and whitespace differ
    'The scheme is "use it or lose it".',                                # straight quotes for curly ones
    "declined to roughly £131m",                                         # a paraphrase
    "lose it”. Only around 4,000 employers",                             # crosses the chunk boundary
]:
    match = matcher.find(quote)
    print(f"{match.status:10} spans={[(s.chunk_id, s.start, s.end) for s in match.spans]}  <- {quote!r}")

**3. What a failed check means.** A claim that fails a check is *rejected* with a reason and sent to the
repair lane (§7) together with the judge's failures. The reasons you will see most often:

| Reason | Meaning |
|---|---|
| `span_not_found` | the claim text is not an exact substring of the prose |
| `quote_not_found` | the quote is not in the cited document's frozen text |
| `unreturned_chunk_id` | the writer cited a chunk it did not read in this section |
| `unappraised_doc_citation` | the cited document has no appraisal, so it is readable but not citable |
| `uncited_finding_id` | the writer cited a finding it did not read in this section (or that is not in its seed) |
| `pattern_mismatch` | the stated numbers do not equal the recomputed numbers |
| `theme_unknown_id` | a theme or group id that does not exist |
| `reasoning_over_cap` | a fourth reasoning claim in one section |
| `substrate_ungated_type` | a claim type that is not available on this run |

One exception is not a rejection: a `corpus_absence` gap without a valid coverage record is **downgraded**
to `inferred` and flagged `gap_degraded`, so the absence claim survives at the grade it can support.

The roll-up counts these outcomes for the whole run:

In [ ]:
c = out["counts"]
pd.Series({
    "claims rejected by a structural check": c["claims_rejected_structural"],
    "chunk claims rejected (quote not found or empty)": c["chunk_claims_rejected"],
    "span bind failures": c["span_bind_failures"],
    "gap claims downgraded to inferred": c["gap_claims_degraded"],
    "citations verified (quote presence)": c["citations_verified"],
    "citations unverified": c["citations_unverified"],
    "citations to unselected (but screened-in) documents": c["citations_from_unselected"],
}).to_frame("count")

## 6. Step 4: the grounding judge (LLM-as-a-judge)

The deterministic checks prove a quote *exists*. They cannot tell whether the sentence built on it is
fair to the source. That is the judge's job: a **separate, cheaper model, with no tools and no memory**, so
the writer never grades its own work.

**One call per block.** Code assembles an *envelope* (`synthesis_envelope_v2`) with:

- the claims to judge: `finding`, `chunk` and `reasoning` claims only (`pattern`, `theme` and `gap` were
  settled by code), each with its citations as resolved by the checks (quote, match status, where it was found);
- the **full frozen text** of every cited chunk, including the chunks that a finding's anchor quotes live in,
  so the judge reads each quote in its context and can spot quote-mining;
- the whole section prose and a map of every claim's span, so the judge reads each claim in context and can
  see which prose is *not* claimed;
- the question and the section focus, as context for reading, explicitly *not* as a reason to be lenient.

**The verdict ladder** (exactly one per claim):

| Verdict | Meaning | Remedy shown to the reader |
|---|---|---|
| `tier_1` | rests on a direct quote that supports the claim as worded | — |
| `tier_2` | a legitimate inference from one cited document | — |
| `tier_3` | legitimate reasoning across several cited sources | — |
| `tier_4` | pure reasoning or background, honest only because it is labelled as such | never counts as evidence |
| `unsupported_mis_cited` | a **failure state, not a tier**: the cited sources do not support the claim as worded (overstatement, dropped caveat, wrong population, fabricated support, quote-mining) | "the cited evidence does not support this" |

Plus two additive outputs: `weakly_grounded` (the claim stands but is thin; orthogonal to the tier) and
`unspanned_assertions` (passages of connective prose that in fact assert evidence; flagged, never edited).
The judge must return exactly one verdict per claim it was given; if coverage is wrong the whole run fails
rather than guess. Every verdict is stored with the judge model, prompt version, envelope version and a hash
of the full input/output, so verdicts can be re-examined later.

In [ ]:
from policy_atlas.evidence_search.synthesis.grounding_judge import ENVELOPE_VERSION, JUDGE_MODEL, JUDGE_PROMPT_VERSION, JUDGE_SYSTEM_PROMPT

print("judge model:", JUDGE_MODEL, "| prompt:", JUDGE_PROMPT_VERSION, "| envelope:", ENVELOPE_VERSION, "\n")
print(JUDGE_SYSTEM_PROMPT)

The judge's first call on our section: what it saw, and what it returned. The `citations` column shows the
deterministic results the judge was handed (`exact` / `normalised` / `failed`), so you can see the two
layers working together: a `failed` anchor is visible to the judge, and the judge then rules the claim
unsupported *as cited*.

In [ ]:
def judge_envelope(judge_obs):
    "Parse the judge's user message back into its labelled parts."
    env = {}
    for part in judge_obs.input["messages"][1]["content"].split("\n\n"):
        label, _, body = part.partition("\n")
        env[label.split(" (")[0].strip().lower()] = json.loads(body)
    return env


def judge_table(judge_obs):
    env = judge_envelope(judge_obs)
    verdicts = {v["claim_id"]: v for v in judge_obs.output["verdicts"]}
    rows = []
    for claim in env["claims to judge"]:
        v = verdicts.get(claim["claim_id"], {})
        rows.append({
            "claim_id": claim["claim_id"],
            "type": claim["claim_type"],
            "claim text": claim["text"][:80] + ("…" if len(claim["text"]) > 80 else ""),
            "citations": ", ".join(c.get("match_status", "?") for c in claim["citations"]) or "(none)",
            "verdict": v.get("verdict"),
            "weak": v.get("weakly_grounded"),
            "rationale": v.get("rationale", "")[:150] + "…",
        })
    return pd.DataFrame(rows), env


judges = [o for o in sec_obs if o.name == "judge:grounding"]
first_judge = judges[0]
table, env = judge_table(first_judge)
print(f"envelope: {len(env['claims to judge'])} claims, {len(env[next(k for k in env if k.startswith('cited chunks'))])} cited chunks' full text, "
      f"{meta(first_judge).get('prompt_tokens'):,} prompt tokens, {seconds(first_judge):.0f} s, ${cost(first_judge):.3f}")
table

Two rationales in full, to see the grading standard: the judge is asked to be "permissive about legitimate
inference, strict about attribution fidelity".

In [ ]:
for v in first_judge.output["verdicts"]:
    if v["verdict"] in ("tier_2", "unsupported_mis_cited"):
        claim = next(c for c in env["claims to judge"] if c["claim_id"] == v["claim_id"])
        print(f"[{v['verdict']}] {claim['text']}")
        print(textwrap.fill("rationale: " + v["rationale"], 110), "\n")

**Unspanned assertions.** The same call reports connective prose that asserts evidence without being a
claim. Each excerpt is bound back into the prose and stored as an `unspanned_assertion` annotation; the
prose is never changed. Experience on this project is that the judge is usually right and the writer
under-anchored, so the fix is in the writer prompt, not the judge.

Code also protects against over-reporting: an excerpt that overlaps a claim span is dropped and counted as
`unspanned_overlap_filtered` (the re-judge after a repair sees only the repaired claims and often re-flags
sentences that are already claims), and an excerpt that cannot be located in the final prose is counted as
`unspanned_unlocated`. Only excerpts that bind cleanly into unclaimed prose are stored.

In [ ]:
shown = 0
for name, os_ in phases.items():
    if not name.startswith("section"):
        continue
    first = next((o for o in os_ if o.name == "judge:grounding"), None)
    for u in (first.output.get("unspanned_assertions") or []) if first else []:
        if shown == 4:
            break
        print(f"[{name}] excerpt  :", textwrap.shorten(u["excerpt"], 150))
        print(f"{' ' * len(name)}   rationale:", textwrap.shorten(u["rationale"], 150), "\n")
        shown += 1
print("in the roll-up:", {k: v for k, v in out["counts"].items() if k.startswith("unspanned")})

Across the whole run: every verdict the judge gave on the body sections, split into the first judgement of
each section and the re-judgement after repair. The unsupported claims are listed with their rationale; a
recurring pattern is "the claim adds a detail the quote does not contain".

In [ ]:
all_verdicts = []
for name, os_ in phases.items():
    judge_calls = [o for o in os_ if o.name == "judge:grounding"]
    for k, j in enumerate(judge_calls):
        env_k = judge_envelope(j)
        texts = {c["claim_id"]: c["text"] for c in env_k["claims to judge"]}
        for v in j.output["verdicts"]:
            all_verdicts.append({"phase": name, "pass": "initial" if k == 0 else "re-judge", **v, "text": texts.get(v["claim_id"], "")})
verdicts_df = pd.DataFrame(all_verdicts)
display(pd.crosstab(verdicts_df["pass"], verdicts_df["verdict"]))

unsupported = verdicts_df[(verdicts_df["verdict"] == "unsupported_mis_cited") & (verdicts_df["pass"] == "initial")]
left = out["counts"]["claims_by_verdict_lane"].get("unsupported_mis_cited", 0)
print(f"{len(unsupported)} claims were ruled unsupported on first judgement; after the repair pass (§7) {left} remain in the final report, flagged.")
print("The first few, with the judge's reason:")
for _, r in unsupported.head(6).iterrows():
    print(f"\n[{r['phase']} / {r['claim_id']}] {textwrap.shorten(r['text'], 150)}")
    print(textwrap.fill("   -> " + r["rationale"], 110))
if len(unsupported) > 6:
    print(f"\n... and {len(unsupported) - 6} more")

## 7. Step 5: one repair pass

Everything that failed in a section is collected: claims rejected by code (§5) and claims the judge
ruled `unsupported_mis_cited` (§6). If the list is empty the section is final. Otherwise there is
**exactly one** repair call, with no tools.

**What the repair call receives**, per failing claim: the claim, the failure reason (the judge's rationale
or the code's reason), the current prose segment and its neighbours, and the *dependency records*: the
full text of the cited chunks or findings the claim already read. It cannot read anything new.

**What it may return**, per failing claim, is one of three things:

1. a rewritten segment carrying a **claim reworded down** to what the evidence supports;
2. a rewritten segment that **makes no evidential assertion** (`claim: null`): the prose stays readable, the claim is gone;
3. an **empty segment**: the sentence is deleted from the prose.

Code splices the segments into the prose, re-binds every span, re-validates the rewritten claims and
re-judges them (one more judge call). There is no second repair. On exhaustion the rules are fixed: a
claim the judge still rules unsupported **stays in the report, flagged** `unsupported_mis_cited` (flag,
don't drop); a claim that still fails a code check is **excluded from the claim list and counted**, its
prose left as the repair wrote it. A repair whose output cannot be parsed, or that answers for the wrong
claim ids, is recorded as such and applies nothing.

In [ ]:
from policy_atlas.evidence_search.synthesis.synthesis_backend import SECTION_REPAIR_TEMPLATE

print(SECTION_REPAIR_TEMPLATE.split("{repair_json}")[1])

In [ ]:
repairs = [o for o in sec_obs if o.name == "synthesise:repair_claims"]
if not repairs:
    print(SECTION, "needed no repair")
else:
    repair = repairs[0]
    content = repair.input["messages"][-1]["content"]          # template text, the JSON, then the instructions
    failing = json.JSONDecoder().raw_decode(content, content.index("{"))[0]["failing_claims"]
    print(f"{len(failing)} failing claim(s) sent for repair ({meta(repair).get('prompt_tokens'):,} prompt tokens, ${cost(repair):.3f}):\n")
    for f in failing:
        deps = f.get("dependencies", {})
        print(f"[{f['claim_id']}] {f['claim']['claim_type']} claim: {textwrap.shorten(f['claim']['text'], 160)}")
        print(textwrap.fill("   failure reason: " + f["failure_reason"], 110))
        print("   dependency records supplied:", {k: (len(v) if isinstance(v, (list, dict)) else 1) for k, v in deps.items()})
        print()
    print("what the model returned:")
    for r in repair.output["repairs"]:
        outcome = ("segment deleted" if r["replacement_segment"] == "" else
                   "assertion removed, prose kept (claim: null)" if r["claim"] is None else "claim reworded down")
        print(f"[{r['claim_id']}] {outcome}")
        if r["replacement_segment"]:
            print(textwrap.fill("   new segment: " + r["replacement_segment"], 110))
        if r["claim"]:
            print("   new claim cites:", r["claim"]["cited_finding_ids"] or [c["chunk_record_id"][:8] for c in r["claim"]["citations"]])
        print()
    if len(judges) > 1:
        print("re-judge of the repaired claims:")
        display(judge_table(judges[1])[0])

Every repair outcome on this run, in one table. "Reworded down" is the primary remedy the design asks for;
deleting a sentence is the last resort.

In [ ]:
rows = []
for name, os_ in phases.items():
    for o in (o for o in os_ if o.name == "synthesise:repair_claims"):
        for r in o.output["repairs"]:
            rows.append({"phase": name, "claim_id": r["claim_id"],
                         "outcome": ("segment deleted" if r["replacement_segment"] == "" else
                                     "assertion removed" if r["claim"] is None else "claim reworded down")})
repairs_df = pd.DataFrame(rows)
display(repairs_df.groupby("outcome").size().to_frame("claims"))
print("flags:", {k: v for k, v in out["flags"].items() if k.startswith("repair")})

## 8. Step 6: after the body sections

Four more things are written once every section (including the injected **Conclusions**) is final.

**Key findings** (`synthesise:write_key_findings`). Written **last, shown first**. The writer receives
the *ledger of surviving claims* of every section, with their verdicts, finding ids and quotes, plus the
text of the cited chunks. It distils 3 to 7 one-line bullets in "claim lead: warrant" form. A key finding may
only re-state what a section established and must cite the **same sources** (never a section). The block is
validated, judged and repaired exactly like a section. If the sections support no headline, an empty block is
correct and nothing is forced.

In [ ]:
kf = next(o for o in obs if o.name == "synthesise:write_key_findings")
print(kf.output["prose"])
kf_judges = [o for o in phases["key findings"] if o.name == "judge:grounding"]
print("claims:", Counter(c["claim_type"] for c in kf.output["claims"]), "| judge verdicts:",
      dict(Counter(v["verdict"] for v in kf_judges[0].output["verdicts"])) if kf_judges else "none",
      "| repairs:", sum(o.name == "synthesise:repair_claims" for o in phases["key findings"]))
print("present in the final report:", out["counts"]["key_findings"])

**Case studies** (`synthesise:write_case_studies`). The writer receives the same ledger plus each cited
document's appraisal, type and year, and returns 2 to 4 *programme cards*: one real programme each (a place
paired with an instrument), 2 to 4 sentences on how it works, claims of type `finding`, `chunk` or
`reasoning` only, and the index of the one claim that states the programme's primary result. Code drops
cards with duplicate titles, a bad result index or empty prose; each surviving card is validated and
judged on its own (no repair; a card whose claims do not survive is dropped). Fewer than two surviving
cards means **no block**, with the reason recorded (`insufficient_programmes` or
`cards_failed_validation`). The whole pass is fail-soft: an error here never fails the run.

In [ ]:
cs = next((o for o in obs if o.name == "synthesise:write_case_studies"), None)
if cs is None:
    print("no case-studies pass on this run")
else:
    for card in cs.output["cards"]:
        print("###", card["title"])
        show_prose(card["prose"])
        result = card["claims"][card["result_ordinal"]] if 0 <= card["result_ordinal"] < len(card["claims"]) else None
        print("   claims:", len(card["claims"]), "| primary result claim:", textwrap.shorten(result["text"], 120) if result else "invalid index", "\n")
    cs_judges = [o for o in phases.get("case studies", []) if o.name == "judge:grounding"]
    print("judge calls (one per surviving card):", len(cs_judges), "| in the final report:", out["counts"]["case_studies"])

**Most-relevant-source notes** (`synthesise:write_source_note`). Code ranks the cited sources by how many
surviving claims cite them and asks the small model for one factual sentence about each of the top three,
restating only facts in the seed. **Full-report intro** (`synthesise:write_intro`): one or two sentences,
from the section titles and briefs, explaining the order of the report. Both are fail-soft.

In [ ]:
for note in out["counts"].get("most_relevant_notes", []):
    print("-", textwrap.fill(note["note"], 105), "\n")
print("intro:", textwrap.fill(out["counts"].get("full_report_intro", ""), 105))

## 9. Step 7: what is saved, and the roll-up

All of this runs inside **one database transaction**. The order of writes is deliberate: the roll-up row
is the very last statement, so a half-finished run leaves blocks but no roll-up, and a re-run of the same
run id is refused before anything is written.

| Table | One row per | What it holds |
|---|---|---|
| `artefact` | report | title (derived from the question), Task, created time |
| `block` | section, plus Key findings and Case studies | the prose, exactly as written, and its content hash |
| `addressable_unit` | claim (and each unspanned assertion) | the span: `{"start": n, "end": m}` into the block, and the text at that span |
| `annotation` | claim | the typed payload: claim type, verdict, weakly-grounded, rationale, resolved citations or anchors, flags, judge model and prompt version, the hash of the judge's input/output |
| `citation` | verified quote | the quote, the chunk it was found in, `verification_result = pass` |
| `synthesis_result` | run | the roll-up: provenance (prompt and model versions, caps, call counts, which upstream runs were used, retrieval settings), per-block counts, run counts, flags |

Every persisted claim is span-anchored by construction: the write checks `prose[start:end] == claim.text`
and fails the run otherwise. The report page is rendered from these rows, so what the reader sees is what
was verified.

**The run-level flags**, which the app and the eval work read:

| Flag | Raised when |
|---|---|
| `unsupported_claims_present` | at least one claim is in the report with verdict `unsupported_mis_cited` |
| `weakly_grounded_present` | at least one claim is flagged weakly grounded (thin support, or a finding whose anchor quote could not be verified) |
| `claims_rejected_structural`, `chunk_claims_rejected`, `span_bind_failed` | some claims were excluded after failing a code check |
| `unspanned_assertions_present` | the judge flagged connective prose that asserts evidence |
| `repair_path_taken` | at least one section went through the repair call |
| `repair_count_mismatch`, `repair_unparseable` | the repair answered for the wrong claim ids, or could not be parsed |
| `groups_unsectioned` | some facet groups were assigned to no section (counted per facet, never silently dropped) |
| `uncited_sections` | a section ended with no cited claim at all |
| `turn_cap_hit` | a section used all its turns and was forced to hand in |

**Hard failures.** A few conditions stop the run and roll back the transaction, with a named reason:
a second execution of the same run (`same_run_reexecution`), no groundable substrate, an invalid steering
directive, a section proposal that is still invalid after its one repair, a judge that did not return one
verdict per claim (`judge_coverage_invalid`), a backend error, and a **generation budget** on the total
number of model calls, computed from the caps so that no loop can run away.

In [ ]:
from policy_atlas.evidence_search.synthesis.synthesise import generation_budget_max

print("generation budget (maximum model calls for one synthesise run):", generation_budget_max())
print("model calls this run actually made (summaries excluded: they run after the commit, outside the budget):")
in_budget = Counter(o.name for o in generations if o.name.startswith(("synthesise:", "judge:")) and not o.name.endswith("_summary"))
print("total:", sum(in_budget.values()))
pd.Series(in_budget).sort_values(ascending=False).to_frame("calls")

## 10. Step 8: navigation summaries, after the commit

Once the transaction has committed, `write_summaries_after_commit` writes a **one-line summary per block**
and a short **"In brief" paragraph for the report**. These sit *outside* the grounding economy: a summary
is a condensed rendering of the detail beneath it, carries no citations and adds no content. Its one
integrity property is faithfulness to that detail, including its epistemic status (a flagged or weak claim
must stay hedged in the summary).

Mechanism: the writer model drafts the summary from the block's prose **with its annotations** (verified,
flagged, weakly grounded and gap spans marked); the same model, in a separate judge call with the summary
judge prompt, returns `pass` or `fail` with a reason; on `fail` the summary is regenerated, at most twice
more; after that it is marked `failed` and the page shows no summary rather than an unverified one. This
runs in its own short transactions so that a provider failure can never roll back the report.

In [ ]:
from policy_atlas.evidence_search.synthesis.synthesise import SUMMARY_REGENERATE_CAP

# Each summary target (a block, or the report) is written then judged; a fail triggers a rewrite, up to the cap.
summary_obs = phases.get("summaries", [])
rows, pending, target, attempt = [], None, 0, 0
for o in summary_obs:
    if o.name in ("synthesise:write_block_summary", "synthesise:write_artefact_summary"):
        pending = {"kind": o.name.split("_")[1], "summary": textwrap.shorten(o.output["summary"], 100)}
    elif o.name == "synthesise:judge_summary" and pending is not None:
        attempt += 1
        rows.append({"target": target, "attempt": attempt, **pending, "verdict": o.output["verdict"], "reason": textwrap.shorten(o.output["reason"], 110)})
        if o.output["verdict"] == "pass" or attempt == SUMMARY_REGENERATE_CAP + 1:
            target, attempt = target + 1, 0
        pending = None
summaries_df = pd.DataFrame(rows)
final = summaries_df.groupby("target").agg(kind=("kind", "first"), attempts=("attempt", "max"), final=("verdict", "last"))
final["final"] = final["final"].map({"pass": "verified", "fail": "failed (no summary shown)"})
print(f"{len(summaries_df)} drafts judged for {len(final)} targets ({out['counts']['blocks_written']} blocks + the report); "
      f"regeneration cap: {SUMMARY_REGENERATE_CAP} retries")
display(final.groupby(["kind", "final"]).size().to_frame("targets"))
summaries_df.head(12)

On this run most first drafts failed the summary judge, and several blocks ended with no verified summary
after their three attempts. That is the design working as specified: a failed summary is shown as absent,
never as an unverified line of text, and the report itself is unaffected. A high failure rate is a signal
about the summariser prompt or the judge's strictness, which is owned by the evaluation work.

## 11. Cost and time of the run

Where the money goes. Section turns dominate because each hand-in turn carries the whole section
conversation (prompt tokens); the cache column shows how much of that the provider served from cache.

In [ ]:
rows = []
for name, os_ in pd.DataFrame({"o": generations}).assign(name=lambda f: f["o"].map(lambda o: o.name)).groupby("name"):
    os_ = list(os_["o"])
    rows.append({
        "observation": name, "calls": len(os_),
        "prompt tokens": sum(int(meta(o).get("prompt_tokens") or 0) for o in os_),
        "cached tokens": sum(int(meta(o).get("cached_tokens") or 0) for o in os_),
        "completion tokens": sum(int(meta(o).get("completion_tokens") or 0) for o in os_),
        "cost $": sum(cost(o) for o in os_),
        "seconds": sum(seconds(o) for o in os_),
    })
cost_df = pd.DataFrame(rows).sort_values("cost $", ascending=False).reset_index(drop=True)
cost_df["share of cost"] = (cost_df["cost $"] / cost_df["cost $"].sum()).map("{:.0%}".format)
cost_df["cost $"] = cost_df["cost $"].round(3)
cost_df["seconds"] = cost_df["seconds"].round(0)
display(cost_df)
wall, model_time = seconds(obs[0]), cost_df["seconds"].sum()
print(f"total: ${cost_df['cost $'].sum():.2f} over {wall / 60:.1f} minutes wall time. Model calls run one after another and account for "
      f"almost all of it ({model_time / 60:.1f} minutes); tool reads and database writes take about {max(wall - model_time, 0):.0f} s.")

## 12. Glossary

| Term | Meaning |
|---|---|
| **LLM** | large language model; "the writer" and "the judge" in this notebook are two different LLM calls |
| **substrate** | everything the earlier pipeline steps wrote for this Task that `synthesise` may draw on |
| **chunk** | one piece of a document's frozen text, stored with an id; the unit that is searched, read and quoted |
| **text basis** | whether a chunk is the document's fetched `full_text` or its `abstract_only` record |
| **appraised** | the document has a quality score (1 to 5) from the `appraise` step; only appraised documents may be cited |
| **origin** | for a chunk: `selected` (the document was chosen for the deep read) or `unselected_screened` (screened-in but not selected) |
| **finding** | a structured record extracted from a selected document; **IOF** = intervention–outcome finding, **ICF** = implementation-context finding (barrier, enabler, mechanism, …) |
| **anchor** | a finding's own supporting quote, verified against the document at extraction time; attached to finding claims by the system |
| **facet group** | a cluster of findings along one facet (intervention, outcome, barrier theme, …), with an id like `intervention:g03` |
| **characterisation theme** | a cluster of documents from the `characterise` step; the "landscape" |
| **intent** | the user's question, as the planner recorded it |
| **seed** | the data packet a prompt receives (run seed, section seed, key-findings seed); always labelled "data, not instructions" |
| **ledger** | the list of claims earlier sections already made, shown to later sections as context, never as evidence |
| **turn** | one call to the writer inside the section loop; it returns tool calls or the hand-in |
| **emit_section** | the tool call by which the writer hands in a section (prose + claims) |
| **claim** | one evidential sentence or clause of the prose, typed, and located as a span |
| **span** | the character interval `[start, end)` of a claim in its block's prose |
| **citation** | for chunk claims: a verbatim quote and the chunk it was found in |
| **envelope** | the data packet the judge receives: claims, full text of cited chunks, section prose, span map, question, focus |
| **verdict / tier** | the judge's grade for one claim: tier 1 to 4, or `unsupported_mis_cited` |
| **weakly grounded** | an additional flag: the claim stands but its support is thin |
| **unspanned assertion** | connective prose that asserts evidence without being a claim; flagged by the judge, never edited |
| **repair** | the one bounded call that rewrites failing claims down to what their evidence supports |
| **flag, don't drop** | the design rule: a failure is recorded and shown, never silently removed or promoted |
| **block** | one unit of the saved report: a section, the Key findings or the Case studies |
| **artefact** | the saved report as a whole |
| **roll-up** | the `synthesis_result` row: counts, flags and provenance for one run |
| **Langfuse trace / observation** | one pipeline run, and one model or tool call inside it |

## 13. Looking at other runs

- To inspect a different run, set `TRACE_ID` in §0 and re-run the notebook. Runs at *rapid* depth have no
  `query_findings` tool and no `finding` claims; runs without `characterise` have no `theme` claims.
- In the Langfuse UI the same structure is visible as a tree: `run:synthesise` → `component:synthesise` →
  the observations named in §0, in time order. Filter traces by metadata `task_id` to see every run of one
  Task.
- To see the saved rows rather than the trace, `pipeline_walkthrough.ipynb` (same folder) runs the whole
  pipeline on a tiny fixture corpus with stub models and queries the tables directly;
  `CONNECT_TO_DATABASE.md` at the repo root explains how to browse the production database.